In [0]:

from pyspark.sql import functions as F

catalog = "workspace"
schema = "default"
volume = "peruvian_food_volume"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"

print(base_path)

In [0]:
restaurantes = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .load(f"{base_path}/restaurantes.csv")
)

reviews = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .load(f"{base_path}/reviews.csv")
)

model = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .load(f"{base_path}/model.csv")
)

In [0]:
restaurantes = spark.read.csv("/Volumes/workspace/default/peruvian_food_volume/restaurants.csv", header=True, inferSchema=True)
reviews = spark.read.csv("/Volumes/workspace/default/peruvian_food_volume/reviews.csv", header=True, inferSchema=True)
model = spark.read.csv("/Volumes/workspace/default/peruvian_food_volume/model.csv", header=True, inferSchema=True)

In [0]:
restaurantes.write.mode("overwrite").format("delta").saveAsTable(
    "workspace.default.restaurantes_raw"
)

reviews.write.mode("overwrite").format("delta").saveAsTable(
    "workspace.default.reviews_raw"
)

model.write.mode("overwrite").format("delta").saveAsTable(
    "workspace.default.model_raw"
)

print("Tabelas RAW criadas corretamente")

**Primeiro teste de qualidade: valores nulos**

Agora vamos medir a completude, que é um dos critérios solicitados.

In [0]:
def null_report(df, table_name):
    print(f"\n===== {table_name} =====")
    
    total = df.count()
    
    for column in df.columns:
        nulls = df.filter(
            F.col(column).isNull() | (F.trim(F.col(column).cast("string")) == "")
        ).count()
        
        percentage = (nulls / total) * 100
        
        print(f"{column}: {nulls} nulos/vacíos ({percentage:.2f}%)")

In [0]:
null_report(restaurantes, "RESTAURANTES")
null_report(reviews, "REVIEWS")
null_report(model, "MODEL")

**Segundo teste: duplicatas**

Agora vamos verificar a unicidade.

In [0]:
print("===== DUPLICADOS RESTAURANTES =====")

duplicados_restaurantes = (
    restaurantes
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicados_restaurantes)

In [0]:
print("===== DUPLICADOS REVIEWS =====")

duplicados_reviews = (
    reviews
    .groupBy("id_review")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicados_reviews)

print("===== DUPLICADOS MODEL =====")

duplicados_model = (
    model
    .groupBy("id_review")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicados_model)

**Terceiro teste: converter os campos numéricos**

Este é um dos pontos importantes que descobrimos graças ao seu printSchema().

In [0]:
restaurantes_clean = (
    restaurantes
    .withColumn("id", F.col("id").cast("long"))
    .withColumn("x", F.col("x").cast("double"))
    .withColumn("y", F.col("y").cast("double"))
    .withColumn("stars", F.col("stars").cast("double"))
    .withColumn("n_reviews", F.col("n_reviews").cast("long"))
    .withColumn("min_price", F.col("min_price").cast("double"))
    .withColumn("max_price", F.col("max_price").cast("double"))
)

In [0]:
reviews_clean = (
    reviews
    .withColumn("score", F.col("score").cast("double"))
    .withColumn("likes", F.col("likes").cast("long"))
    .withColumn("service", F.col("service").cast("long"))
)

In [0]:
reviews_clean = (
    reviews
    .withColumn("score", F.col("score").cast("double"))
    .withColumn("likes", F.col("likes").cast("long"))
    .withColumn("service", F.col("service").cast("long"))
)

In [0]:
model_clean = model

**Verificar os novos tipos**

In [0]:
restaurantes_clean.printSchema()

In [0]:
reviews_clean.printSchema()

**Qualidade dos preços**

Vamos verificar se existem preços negativos ou inconsistentes:

**Um teste MUITO importante: relações entre arquivos**

restaurantes.id
      │
      │
      ▼
reviews.service
      │
      │
      ▼
model.id_review

In [0]:
total_model = model_clean.count()

model_sin_review = (
    model_clean
    .join(
        reviews_clean.select("id_review"),
        on="id_review",
        how="left"
    )
    .filter(reviews_clean.id_review.isNull())
    .count()
)

print("Total registros model:", total_model)
print("Registros de modelo sem avaliação associada:", model_sin_review)

In [0]:
reviews_clean = (
    reviews
    .withColumn(
        "score_clean",
        F.expr("try_cast(trim(score) AS double)")
    )
    .withColumn(
        "likes_clean",
        F.expr("try_cast(trim(likes) AS long)")
    )
    .withColumn(
        "service_clean",
        F.expr("try_cast(trim(service) AS double)")
    )
)

In [0]:
restaurantes_clean = (
    restaurantes
    .withColumn(
        "id_clean",
        F.expr("try_cast(trim(cast(id AS string)) AS long)")
    )
    .withColumn(
        "x_clean",
        F.expr("try_cast(trim(x) AS double)")
    )
    .withColumn(
        "y_clean",
        F.expr("try_cast(trim(y) AS double)")
    )
    .withColumn(
        "stars_clean",
        F.expr("try_cast(trim(stars) AS double)")
    )
    .withColumn(
        "n_reviews_clean",
        F.expr("try_cast(trim(n_reviews) AS double)")
    )
    .withColumn(
        "min_price_clean",
        F.expr("try_cast(trim(min_price) AS double)")
    )
    .withColumn(
        "max_price_clean",
        F.expr("try_cast(trim(max_price) AS double)")
    )
)

In [0]:
reviews_clean = (
    reviews_clean
    .withColumn(
        "restaurant_id",
        F.expr("try_cast(service_clean AS long)")
    )
)

**Agora, vamos detectar as pontuações inválidas**

In [0]:
scores_invalidos = (
    reviews_clean
    .filter(
        F.col("score_clean").isNull() &
        F.col("score").isNotNull()
    )
)

print("Scores que no pudieron convertirse:", scores_invalidos.count())

display(
    scores_invalidos
    .select(
        "id_review",
        "score",
        "review",
        "title"
    )
    .limit(20)
)

**Agora sim: distribuição do score**

In [0]:
print("===== DISTRIBUCIÓN DE SCORE =====")

display(
    reviews_clean
    .filter(F.col("score_clean").isNotNull())
    .groupBy("score_clean")
    .count()
    .orderBy("score_clean")
)

In [0]:
print("===== Pontuações fora da faixa de 1 a 5 =====")

display(
    reviews_clean
    .filter(
        F.col("score_clean").isNotNull() &
        (
            (F.col("score_clean") < 1) |
            (F.col("score_clean") > 5)
        )
    )
    .select(
        "id_review",
        "score",
        "score_clean"
    )
    .limit(50)
)

In [0]:
print("===== SCORES FUERA DEL RANGO 1-5 =====")

display(
    reviews_clean
    .filter(
        F.col("score_clean").isNotNull() &
        (
            (F.col("score_clean") < 1) |
            (F.col("score_clean") > 5)
        )
    )
    .select(
        "id_review",
        "score",
        "score_clean"
    )
    .limit(50)
)

In [0]:
print("===== PREÇOS NEGATIVOS =====")

display(
    restaurantes_clean
    .filter(
        (F.col("min_price_clean") < 0) |
        (F.col("max_price_clean") < 0)
    )
    .select(
        "id",
        "name",
        "min_price",
        "min_price_clean",
        "max_price",
        "max_price_clean"
    )
)

In [0]:
print("===== MIN_PRICE > MAX_PRICE =====")

display(
    restaurantes_clean
    .filter(
        F.col("min_price_clean").isNotNull() &
        F.col("max_price_clean").isNotNull() &
        (F.col("min_price_clean") > F.col("max_price_clean"))
    )
    .select(
        "id",
        "name",
        "min_price",
        "max_price",
        "min_price_clean",
        "max_price_clean"
    )
)

In [0]:
print("Total reviews:", reviews.count())

print(
    "Reviews con score original no numérico:",
    reviews_clean
    .filter(
        F.expr("try_cast(trim(score) AS double)").isNull() &
        F.col("score").isNotNull()
    )
    .count()
)

In [0]:
display(
    reviews_clean
    .filter(
        F.expr("try_cast(trim(score) AS double)").isNull() &
        F.col("score").isNotNull()
    )
    .select(
        "id_review",
        "review",
        "title",
        "score",
        "likes",
        "service",
        "date",
        "platform"
    )
    .limit(10)
)

**NOVA EXTRAÇÃO COM DADOS LIMPOS**

In [0]:
reviews_clean = (
    reviews
    .withColumn(
        "score_clean",
        F.expr("try_cast(trim(score) AS double)")
    )
    .withColumn(
        "likes_clean",
        F.expr("try_cast(trim(likes) AS double)")
    )
    .withColumn(
        "service_clean",
        F.expr("try_cast(trim(service) AS double)")
    )
    .withColumn(
        "restaurant_id",
        F.expr("try_cast(service_clean AS long)")
    )
)

In [0]:
reviews_invalidos = (
    reviews_clean
    .filter(
        F.col("score_clean").isNull() |
        ~F.col("score_clean").between(1, 5)
    )
)

print("Registros excluidos por score:", reviews_invalidos.count())

In [0]:
reviews_clean = (
    reviews_clean
    .filter(
        F.col("score_clean").isNotNull() &
        F.col("score_clean").between(1, 5)
    )
)

In [0]:
print("Reviews originales:", reviews.count())
print("Reviews válidas:", reviews_clean.count())
print("Reviews excluidas:", reviews.count() - reviews_clean.count())

In [0]:
display(
    reviews_clean
    .groupBy("score_clean")
    .count()
    .orderBy("score_clean")
)

In [0]:
reviews_clean.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.default.reviews_clean")

In [0]:
restaurantes_clean.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.default.restaurantes_clean")

In [0]:
model_clean.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.default.model_clean")

In [0]:
print("Reviews originales:", reviews.count())
print("Reviews válidas:", reviews_clean.count())
print("Reviews excluidas:", reviews.count() - reviews_clean.count())